In [7]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
working_directory = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "src" / "procesado" / "limpiar_poblacion.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/procesado.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 1. Cleaning population

Read the Andalusian population Bronze table, apply the current cleaning steps, and preview the result.

In [ ]:
from src.procesado.limpiar_poblacion import clean_population

population_bronze_path = project_root / "data" / "raw" / "population_andalusia.csv"
population_bronze = pd.read_csv(
    population_bronze_path,
    dtype=str,
    keep_default_na=False,
)
population_cleaned = clean_population(population_bronze)
print(f"Bronze rows: {len(population_bronze):,}; cleaned rows: {len(population_cleaned):,}")
display(population_cleaned.head(10))

Bronze rows: 1,201,050; cleaned rows: 129,525


,municipality,province,province_code,age,population,year,source,downloaded_at_utc
0,Abla,Almería,04,18,15,2025,INE,2026-10-19T07:07:30.076995+00:00
1,Abla,Almería,04,19,11,2025,INE,2026-10-19T07:07:30.076995+00:00
2,Abla,Almería,04,20,11,2025,INE,2026-10-19T07:07:30.076995+00:00
3,Abla,Almería,04,21,8,2025,INE,2026-10-19T07:07:30.076995+00:00
4,Abla,Almería,04,22,17,2025,INE,2026-10-19T07:07:30.076995+00:00
5,Abla,Almería,04,23,9,2025,INE,2026-10-19T07:07:30.076995+00:00
6,Abla,Almería,04,24,16,2025,INE,2026-10-19T07:07:30.076995+00:00
7,Abla,Almería,04,25,9,2025,INE,2026-10-19T07:07:30.076995+00:00
8,Abla,Almería,04,26,13,2025,INE,2026-10-19T07:07:30.076995+00:00
9,Abla,Almería,04,27,12,2025,INE,2026-10-19T07:07:30.076995+00:00


# 2. Cleaning income

Read the Andalusian income Bronze table, normalize it for joining with population, and preview the result.

In [2]:
from src.procesado.limpiar_renta import clean_income

income_bronze_path = project_root / "data" / "raw" / "income_andalusia.csv"
income_bronze = pd.read_csv(
    income_bronze_path,
    dtype=str,
    keep_default_na=False,
)
income_cleaned = clean_income(income_bronze)
print(f"Bronze rows: {len(income_bronze):,}; cleaned rows: {len(income_cleaned):,}")
display(income_cleaned.head(10))

Bronze rows: 786; cleaned rows: 786


,municipality,province,province_code,mean_net_income_per_person,year,source,downloaded_at_utc
0,Abla,Almería,04,12411.0,2023,INE,2026-10-19T07:09:25.321278+00:00
1,Abrucena,Almería,04,11944.0,2023,INE,2026-10-19T07:09:25.321278+00:00
2,Adra,Almería,04,10156.0,2023,INE,2026-10-19T07:09:25.321278+00:00
3,Albanchez,Almería,04,12035.0,2023,INE,2026-10-19T07:09:25.321278+00:00
4,Alboloduy,Almería,04,11557.0,2023,INE,2026-10-19T07:09:25.321278+00:00
5,Albox,Almería,04,10742.0,2023,INE,2026-10-19T07:09:25.321278+00:00
6,Alcolea,Almería,04,11126.0,2023,INE,2026-10-19T07:09:25.321278+00:00
7,Alcudia de Monteagud,Almería,04,13871.0,2023,INE,2026-10-19T07:09:25.321278+00:00
8,Alcóntar,Almería,04,13927.0,2023,INE,2026-10-19T07:09:25.321278+00:00
9,Alhabia,Almería,04,10670.0,2023,INE,2026-10-19T07:09:25.321278+00:00


# 3. Cleaning gyms

Read the Andalusian OSM Bronze gym table, clean it, save the GeoParquet Silver table, and preview the result.

In [8]:
from src.procesado.limpiar_gimnasios import clean_gyms

gyms_bronze_path = project_root / "data" / "raw" / "gyms_andalusia.csv"
gyms_bronze = pd.read_csv(
    gyms_bronze_path,
    dtype=str,
    keep_default_na=False,
)
gyms_cleaned, sports_facilities_cleaned = clean_gyms(
    gyms_bronze,
    output_mode="both",
)

gyms_silver_path = (
    project_root
    / "data"
    / "processed"
    / "silver"
    / "gyms_andalusia.parquet"
)
gyms_silver_path.parent.mkdir(parents=True, exist_ok=True)
gyms_cleaned.to_parquet(gyms_silver_path, index=False)

sports_facilities_silver_path = (
    project_root
    / "data"
    / "processed"
    / "silver"
    / "sports_facilities_andalusia.parquet"
)
sports_facilities_silver_path.parent.mkdir(parents=True, exist_ok=True)
sports_facilities_cleaned.to_parquet(
    sports_facilities_silver_path,
    index=False,
)
print(
    f"Bronze rows: {len(gyms_bronze):,}; "
    f"gyms: {len(gyms_cleaned):,}; "
    f"sports facilities: {len(sports_facilities_cleaned):,}"
)
print(f"Gyms saved to: {gyms_silver_path}")
print(f"Sports facilities saved to: {sports_facilities_silver_path}")
display(gyms_cleaned.head(10))
display(sports_facilities_cleaned.head(10))

Bronze rows: 3,567; gyms: 564; sports facilities: 3,567
Gyms saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\gyms_andalusia.parquet
Sports facilities saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\sports_facilities_andalusia.parquet


,osm_type,osm_id,province_code,province,latitude,longitude,source,downloaded_at_utc,tag_name,tag_leisure,tag_sport,tag_amenity,geometry
0,node,4512584749,ES-AL,Almería,36.838962,-2.460986,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,e-fitness,fitness_centre,<NA>,<NA>,POINT (-2.46099 36.83896)
1,node,4512664045,ES-AL,Almería,36.839466,-2.456743,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,BOX GYM,fitness_centre,<NA>,<NA>,POINT (-2.45674 36.83947)
2,node,4708078896,ES-AL,Almería,36.696758,-2.789563,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Be in Shape,fitness_centre,<NA>,<NA>,POINT (-2.78956 36.69676)
3,node,5319369078,ES-AL,Almería,37.352822,-2.183695,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Gimnasio municipal,fitness_centre,<NA>,<NA>,POINT (-2.18369 37.35282)
4,node,5966293045,ES-AL,Almería,36.842693,-2.455705,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Activaclub Almería,fitness_centre,<NA>,<NA>,POINT (-2.45571 36.84269)
5,node,6073162803,ES-AL,Almería,36.853818,-2.355312,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Gimnasio Sportgym,fitness_centre,<NA>,<NA>,POINT (-2.35531 36.85382)
6,node,6218335433,ES-AL,Almería,36.843306,-2.443812,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pilates Center Almería,fitness_centre,<NA>,<NA>,POINT (-2.44381 36.84331)
7,node,7146553466,ES-AL,Almería,37.388721,-2.143269,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,"Centro de Ocio ""Kronos""",fitness_centre,<NA>,<NA>,POINT (-2.14327 37.38872)
8,node,9306192196,ES-AL,Almería,36.81332,-2.574834,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Nexa Fit Aguadulce,fitness_centre,<NA>,<NA>,POINT (-2.57483 36.81332)
9,node,9372949624,ES-AL,Almería,37.186776,-1.82007,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Gimnasio Proa,fitness_centre,<NA>,<NA>,POINT (-1.82007 37.18678)


,osm_type,osm_id,province_code,province,latitude,longitude,source,downloaded_at_utc,tag_name,tag_leisure,tag_sport,tag_amenity,geometry
0,node,260812441,ES-AL,Almería,37.172028,-1.833994,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Go-Karting,sports_centre,karting,<NA>,POINT (-1.83399 37.17203)
1,node,1620487480,ES-AL,Almería,37.345363,-2.432541,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellón,sports_centre,<NA>,<NA>,POINT (-2.43254 37.34536)
2,node,1912986759,ES-AL,Almería,37.212201,-1.810827,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Vera Costa,fitness_station,fitness,<NA>,POINT (-1.81083 37.2122)
3,node,2452253961,ES-AL,Almería,36.838087,-2.949279,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellon de deportes,sports_centre,<NA>,<NA>,POINT (-2.94928 36.83809)
4,node,4432638479,ES-AL,Almería,36.969974,-2.607533,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Frontón de Huécija,sports_centre,pelota,<NA>,POINT (-2.60753 36.96997)
5,node,4512584749,ES-AL,Almería,36.838962,-2.460986,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,e-fitness,fitness_centre,<NA>,<NA>,POINT (-2.46099 36.83896)
6,node,4512664045,ES-AL,Almería,36.839466,-2.456743,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,BOX GYM,fitness_centre,<NA>,<NA>,POINT (-2.45674 36.83947)
7,node,4708078896,ES-AL,Almería,36.696758,-2.789563,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Be in Shape,fitness_centre,<NA>,<NA>,POINT (-2.78956 36.69676)
8,node,5319369078,ES-AL,Almería,37.352822,-2.183695,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Gimnasio municipal,fitness_centre,<NA>,<NA>,POINT (-2.18369 37.35282)
9,node,5319369079,ES-AL,Almería,37.352754,-2.183941,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellón Municipal de Deportes,sports_centre,<NA>,<NA>,POINT (-2.18394 37.35275)
